# Wikimedia Dump Converter

This Colab notebook installs the converter as a package and exposes a small widget UI for converting single Wikimedia dumps or dump catalog URLs.

In [ ]:
# @title
import re
import subprocess
import sys

GITHUB_REPO = "https://github.com/0xEodum/WikiParser.git"
INSTALL_S3_EXTRA = False

def clean_github_repo_url(value):
    text = str(value).strip()
    match = re.search(r"https://github\.com/[^\]\)\s]+", text)
    if match:
        text = match.group(0)
    text = text.rstrip("/.)]")
    if not text.startswith("https://github.com/"):
        raise ValueError(f"Expected a GitHub HTTPS URL, got: {value!r}")
    if not text.endswith(".git"):
        text = f"{text}.git"
    return text

repo_url = clean_github_repo_url(GITHUB_REPO)
spec = f"wiki-dump-converter[s3] @ git+{repo_url}" if INSTALL_S3_EXTRA else f"git+{repo_url}"
print(f"Installing from: {spec}")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", spec, "ipywidgets"])

In [ ]:
#@title Настройки конвертации { display-mode: "form" }
#@markdown ### Основные параметры
source = "https://dumps.wikimedia.org/other/mediawiki_content_current/enwiki/2026-05-01/xml/bzip2/" #@param {type:"string"}
output_dir = "/content/wiki_output" #@param {type:"string"}

#@markdown ---
#@markdown ### Настройки парсинга (0 = без лимита)
pages_per_file = 1 #@param {type:"integer"}
limit = 0 #@param {type:"integer"}
archive_limit = 0 #@param {type:"integer"}
namespaces = "0" #@param {type:"string"}
lead_title = "Introduction" #@param {type:"string"}
force_catalog = True #@param {type:"boolean"}
include_redirects = False #@param {type:"boolean"}

#@markdown ---
#@markdown ### Формат и хранилище
output_format = "json" #@param ["json", "ontology"]
storage = "local" #@param ["local", "s3", "both"]

#@markdown ---
#@markdown ### Настройки S3 (заполните, если используется s3 или both)
s3_bucket = "" #@param {type:"string"}
s3_prefix = "" #@param {type:"string"}
s3_endpoint = "" #@param {type:"string"}
s3_region = "" #@param {type:"string"}

# ==========================================
# Исполняемый код начинается здесь
# ==========================================
from pathlib import Path
import json

from wikiparser.dump_converter import convert_catalog, convert_dump, is_catalog_source, parse_namespaces
from wikiparser.s3_io import load_s3_config, upload_directory_to_s3

def optional_int(val):
    return val if val and val > 0 else None

# Собираем аргументы
kwargs = dict(
    output_dir=output_dir,
    pages_per_file=pages_per_file,
    lead_title=lead_title,
    namespaces=parse_namespaces(namespaces),
    skip_redirects=not include_redirects,
    limit=optional_int(limit),
    output_format=output_format,
)

print("Начинаю конвертацию...")

if force_catalog or is_catalog_source(source):
    summary = convert_catalog(
        catalog_url=source,
        archive_limit=optional_int(archive_limit),
        **kwargs,
    )
    print(f"Записано {summary.pages_written} страниц из {summary.archives_processed}/{summary.archives_seen} архивов")
else:
    summary = convert_dump(source=source, **kwargs)
    print(f"Записано {summary.pages_written} страниц")

if storage in {"s3", "both"}:
    config = load_s3_config(
        bucket=s3_bucket or None,
        prefix=s3_prefix,
        endpoint_url=s3_endpoint or None,
        region_name=s3_region or None,
    )
    s3_summary = upload_directory_to_s3(summary.output_dir, config)
    print(f"Загружено {s3_summary.files_uploaded} файлов в s3://{s3_summary.bucket}/{s3_summary.prefix}")

json_files = sorted(Path(summary.output_dir).rglob("*.json"))
print(f"\nДиректория вывода: {summary.output_dir}")
print(f"Количество JSON файлов: {len(json_files)}")

if json_files:
    sample = json.loads(json_files[0].read_text(encoding="utf-8"))
    print(f"\nПример из первого файла ({json_files[0]}):")
    print(json.dumps(sample[0] if isinstance(sample, list) else sample, ensure_ascii=False, indent=2)[:1500])

In [ ]:
import shutil

zip_path = shutil.make_archive("/content/wiki_output", "zip", output_dir.value)
print(f"Created archive: {zip_path}")